In [1]:
!pip install dagshub mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("XGBoost")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ XGBoost version:", xgb.__version__)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1dc472c5-3ee5-4655-8f90-b294a81fcc31&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7c8ae4efb6eeb55771dc8f266663b92927305632d5621c379b201b174928d4ec




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ XGBoost version: 3.2.0


# **Cleaning**

In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train_xgb = train["isFraud"].copy()
    X_train_xgb = train.drop(columns=["isFraud", "TransactionID"])
    X_test_xgb  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train_xgb.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_xgb.columns if c not in id_like_cols]

    missing_ratio = X_train_xgb.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train_xgb = X_train_xgb.drop(columns=drop_missing)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in drop_missing if c in X_test_xgb.columns])

    near_constant_cols = []
    for col in X_train_xgb.columns:
        top_freq = X_train_xgb[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_xgb = X_train_xgb.drop(columns=near_constant_cols)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in near_constant_cols if c in X_test_xgb.columns])

    for col in X_train_xgb.columns:
        if col not in X_test_xgb.columns:
            X_test_xgb[col] = np.nan
    X_test_xgb = X_test_xgb[X_train_xgb.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train_xgb.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test_xgb.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train_xgb.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train_xgb.mean()))

    print(f"X_train: {X_train_xgb.shape}")
    print(f"X_test:  {X_test_xgb.shape}")
    print(f"Fraud rate: {y_train_xgb.mean():.4f}")

    X_train_clean_xgb = X_train_xgb
    X_test_clean_xgb  = X_test_xgb
    y_train_clean_xgb = y_train_xgb

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run XGBoost_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/67c147d26c06440db4bf035062ddf6fe
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X_train = X_train_clean_xgb.copy()
    X_test  = X_test_clean_xgb.copy()
    y_train = y_train_clean_xgb.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train["day_of_week"] = (X_train["TransactionDT"] // 86400) % 7
    X_test["day_of_week"]  = (X_test["TransactionDT"]  // 86400) % 7

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)
    
    group_cols = ["card1", "card2", "card3", "addr1", "P_emaildomain", "R_emaildomain"]
    agg_features_added = 0

    for col in group_cols:
        if col not in X_train.columns:
            continue

        fraud_mean = y_train.groupby(X_train[col]).mean()
        X_train[f"{col}_fraud_rate"]      = X_train[col].map(fraud_mean).fillna(y_train.mean())
        X_test[f"{col}_fraud_rate"]       = X_test[col].map(fraud_mean).fillna(y_train.mean())

        amt_mean = X_train.groupby(col)["TransactionAmt"].mean()
        X_train[f"{col}_amt_mean"]        = X_train[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())
        X_test[f"{col}_amt_mean"]         = X_test[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())

        freq = X_train[col].value_counts()
        X_train[f"{col}_freq"]            = X_train[col].map(freq).fillna(0)
        X_test[f"{col}_freq"]             = X_test[col].map(freq).fillna(0)

        agg_features_added += 3

    if "card1_amt_mean" in X_train.columns:
        X_train["amt_vs_card1_mean"] = X_train["TransactionAmt"] - X_train["card1_amt_mean"]
        X_test["amt_vs_card1_mean"]  = X_test["TransactionAmt"]  - X_test["card1_amt_mean"]
        agg_features_added += 1

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("cat_encoding",          "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform",     True)
    mlflow.log_param("cyclic_time_encoding",  True)
    mlflow.log_param("day_of_week",           True)
    mlflow.log_param("aggregation_features",  True)
    mlflow.log_param("agg_group_cols",        str(group_cols))
    mlflow.log_metric("features_after_fe",    int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",     len(cat_cols))
    mlflow.log_metric("agg_features_added",   agg_features_added)

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")
    print(f"Aggregation features added: {agg_features_added}")

    X_train_fe_xgb = X_train
    X_test_fe_xgb  = X_test
    y_train_fe_xgb = y_train

X_train_fe: (590540, 375)
X_test_fe:  (506691, 375)
Aggregation features added: 19
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/7c459525d9fc4330b2c63311e2bf6374
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Selection

In [5]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    X_train = X_train_fe_xgb.copy()
    X_test  = X_test_fe_xgb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    sample_n = min(120_000, len(X_train))
    idx  = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    protected_cols = [c for c in drop_corr if any(
        kw in c for kw in ["_fraud_rate", "_amt_mean", "_freq", "amt_vs_"]
    )]
    drop_corr = [c for c in drop_corr if c not in protected_cols]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold",          CORR_THRESHOLD)
    mlflow.log_param("protected_agg_features",  len(protected_cols))
    mlflow.log_metric("dropped_const",           len(const_cols))
    mlflow.log_metric("dropped_high_corr",       len(drop_corr))
    mlflow.log_metric("features_after_fs",       int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")
    print(f"Protected aggregation features: {len(protected_cols)}")

    X_train_final_xgb = X_train
    X_test_final_xgb  = X_test

X_train_fs: (590540, 321)
X_test_fs:  (506691, 321)
Protected aggregation features: 0
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/06b42a9067ff4b289afa0bca9ee9f71a
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Training

In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
spw = round(neg / pos, 2)
print(f"Train size: {X_tr.shape} | Val size: {X_val.shape}")
print(f"scale_pos_weight = {spw}  (neg/pos = {neg}/{pos})")

Train size: (472432, 321) | Val size: (118108, 321)
scale_pos_weight = 27.58  (neg/pos = 455902/16530)


In [7]:
with mlflow.start_run(run_name="XGB_Baseline"):
    mlflow.log_params({
        "n_estimators": 100, "max_depth": 6, "learning_rate": 0.3,
        "scale_pos_weight": 1, "tree_method": "hist", "device": "cuda",
        "note": "default params, no class balancing"
    })

    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.3,
        scale_pos_weight=1, tree_method="hist", device="cuda",
        eval_metric="auc", random_state=42
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metrics({
        "train_auc": round(train_auc, 5),
        "val_auc":   round(val_auc,   5),
        "overfit_gap": round(gap, 5)
    })

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

[Baseline] Train: 0.9723 | Val: 0.9593 | Gap: 0.0130
🏃 View run XGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/88a566bca4254968a705580bef5ebd45
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [8]:
for n_est in [100, 200, 400, 800, 1600]:
    with mlflow.start_run(run_name=f"XGB_nestimators_{n_est}"):
        clf = xgb.XGBClassifier(
            n_estimators=n_est,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter  = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred  = clf.predict_proba(X_tr,  iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr,  y_tr_pred)
        val_auc   = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_params({"n_estimators": n_est, "max_depth": 6, "learning_rate": 0.1,
                           "scale_pos_weight": spw, "tree_method": "hist", "device": "cuda"})
        mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                            "overfit_gap": round(gap, 5), "best_iteration": best_iter})

        converged = "✅ CONVERGED" if best_iter < n_est - 1 else "⏩ STILL LEARNING"
        status    = "OVERFIT" if gap > 0.025 else ("UNDERFIT" if val_auc < 0.90 else "OK")
        print(f"  n_est={n_est:<5} | Best Iter: {best_iter:<4} | Val: {val_auc:.4f} | Gap: {gap:.4f} [{status}] {converged}")

  n_est=100   | Best Iter: 99   | Val: 0.9488 | Gap: 0.0092 [OK] ⏩ STILL LEARNING
🏃 View run XGB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/129af957918b4af5ac73fe9e8bbb6185
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_est=200   | Best Iter: 199  | Val: 0.9585 | Gap: 0.0136 [OK] ⏩ STILL LEARNING
🏃 View run XGB_nestimators_200 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/d264c4562f2e4099a778e601b9b8bebe
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_est=400   | Best Iter: 399  | Val: 0.9672 | Gap: 0.0184 [OK] ⏩ STILL LEARNING
🏃 View run XGB_nestimators_400 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/f6384eb1215e4cdb8bd91b94be89b0a0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
 

In [9]:
depth_results = []
for depth in [4, 6, 7, 8, 10]:
    for mcw in [1, 50]:
        run_label = f"depth{depth}_mcw{mcw}"
        with mlflow.start_run(run_name=f"XGB_{run_label}"):
            clf = xgb.XGBClassifier(
                n_estimators=800,
                max_depth=depth,
                min_child_weight=mcw,
                learning_rate=0.1,
                scale_pos_weight=spw,
                tree_method="hist",
                device="cuda",
                eval_metric="auc",
                early_stopping_rounds=50,
                random_state=42
            )
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            best_iter  = clf.best_iteration
            y_tr_pred  = clf.predict_proba(X_tr,  iteration_range=(0, best_iter + 1))[:, 1]
            y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
            train_auc = roc_auc_score(y_tr,  y_tr_pred)
            val_auc   = roc_auc_score(y_val, y_val_pred)
            gap = train_auc - val_auc
            mlflow.log_params({"max_depth": depth, "min_child_weight": mcw,
                               "n_estimators_max": 800, "learning_rate": 0.1})
            mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                                "overfit_gap": round(gap, 5), "best_iteration": best_iter})
            converged = "✅ CONVERGED" if best_iter < 799 else "⏩ STILL LEARNING"
            diagnosis = "OVERFIT" if gap > 0.035 else ("UNDERFIT" if val_auc < 0.91 else "OK")
            depth_results.append({
                "max_depth": depth, "min_child_weight": mcw,
                "train_auc": train_auc, "val_auc": val_auc, "gap": gap, "best_iter": best_iter
            })
            print(f"  {run_label:<20} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f} — {diagnosis} {converged}")

depth_df = pd.DataFrame(depth_results)

converged_runs = depth_df[depth_df["best_iter"] < 799]
if len(converged_runs) == 0:
    converged_runs = depth_df

converged_runs = converged_runs.copy()
converged_runs["stability_score"] = (
    converged_runs["val_auc"] - 2 * (converged_runs["gap"] - 0.020).clip(lower=0)
)

best_row   = converged_runs.loc[converged_runs["stability_score"].idxmax()]
best_depth = int(best_row["max_depth"])
best_mcw   = int(best_row["min_child_weight"])

print(f"\n{'run':<20} | {'val_auc':>8} | {'gap':>6} | {'stability':>10} | converged")
print("-" * 65)
for _, row in converged_runs.sort_values("stability_score", ascending=False).iterrows():
    print(f"  depth{int(row['max_depth'])}_mcw{int(row['min_child_weight']):<13} | "
          f"{row['val_auc']:>8.4f} | {row['gap']:>6.4f} | {row['stability_score']:>10.4f} | ✅")

print(f"\n✅ Best (stability-focused): depth={best_depth}, min_child_weight={best_mcw}")
print(f"   Val AUC={best_row['val_auc']:.4f} | Gap={best_row['gap']:.4f} | Stability Score={best_row['stability_score']:.4f}")

  depth4_mcw1          | Best Iter: 799 | Val: 0.9629 | Gap: 0.0143 — OK ⏩ STILL LEARNING
🏃 View run XGB_depth4_mcw1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/d8919e0b480642a68f32e2e3da436385
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth4_mcw50         | Best Iter: 799 | Val: 0.9629 | Gap: 0.0139 — OK ⏩ STILL LEARNING
🏃 View run XGB_depth4_mcw50 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/36c55173aa5a43d78fc45afefd2e1540
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth6_mcw1          | Best Iter: 798 | Val: 0.9734 | Gap: 0.0213 — OK ✅ CONVERGED
🏃 View run XGB_depth6_mcw1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/4deeaa9ba7c141e393c51c1134953caf
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experim

In [10]:
sampling_results = []
for subsample, colsample in [(1.0, 1.0), (0.8, 0.8), (0.7, 0.7), (0.8, 0.6)]:
    label = f"sub{subsample}_col{colsample}"
    with mlflow.start_run(run_name=f"XGB_sampling_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=800,
            max_depth=best_depth,
            min_child_weight=best_mcw,
            learning_rate=0.1,
            subsample=subsample,
            colsample_bytree=colsample,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        best_iter  = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred  = clf.predict_proba(X_tr,  iteration_range=(0, best_iter + 1))[:, 1]
        train_auc  = roc_auc_score(y_tr,  y_tr_pred)
        val_auc    = roc_auc_score(y_val, y_val_pred)
        gap        = train_auc - val_auc
        mlflow.log_params({"subsample": subsample, "colsample_bytree": colsample,
                           "n_estimators_max": 800})
        mlflow.log_metrics({"train_auc": train_auc, "val_auc": val_auc,
                            "gap": gap, "best_iteration": best_iter})
        sampling_results.append({
            "label": label, "sub": subsample, "col": colsample,
            "val_auc": val_auc, "gap": gap, "best_iter": best_iter  # დაემატა
        })
        converged = "✅ CONVERGED" if best_iter < 799 else "⏩ STILL LEARNING"
        print(f"  {label:<20} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f} {converged}")

samp_df = pd.DataFrame(sampling_results)

converged_samp = samp_df[samp_df["best_iter"] < 799]
if len(converged_samp) == 0:
    converged_samp = samp_df

converged_samp = converged_samp.copy()
converged_samp["stability_score"] = (
    converged_samp["val_auc"] - 2 * (converged_samp["gap"] - 0.020).clip(lower=0)
)

best_samp_row  = converged_samp.loc[converged_samp["stability_score"].idxmax()]
best_subsample = best_samp_row["sub"]
best_colsample = best_samp_row["col"]

print(f"\n{'label':<22} | {'val_auc':>8} | {'gap':>6} | {'stability':>10} | converged")
print("-" * 65)
for _, row in converged_samp.sort_values("stability_score", ascending=False).iterrows():
    print(f"  {row['label']:<20} | {row['val_auc']:>8.4f} | {row['gap']:>6.4f} | {row['stability_score']:>10.4f} | ✅")

print(f"\n✅ Best sampling: subsample={best_subsample}, colsample={best_colsample}")
print(f"   Val AUC={best_samp_row['val_auc']:.4f} | Gap={best_samp_row['gap']:.4f} | Stability={best_samp_row['stability_score']:.4f}")

  sub1.0_col1.0        | Best Iter: 794 | Val: 0.9784 | Gap: 0.0212 ✅ CONVERGED
🏃 View run XGB_sampling_sub1.0_col1.0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/5b750d3f37a04e118d48cae011ba5570
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.8_col0.8        | Best Iter: 774 | Val: 0.9782 | Gap: 0.0216 ✅ CONVERGED
🏃 View run XGB_sampling_sub0.8_col0.8 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/51a13eba4f2449bb86662293dba1868e
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.7_col0.7        | Best Iter: 768 | Val: 0.9780 | Gap: 0.0217 ✅ CONVERGED
🏃 View run XGB_sampling_sub0.7_col0.7 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/3a4ffea432644b53b2c60129463137b5
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/

In [11]:
reg_results = []
for alpha, lam, gamma in [(0, 1, 0), (1, 1, 0), (0, 5, 0), (1, 5, 0), (1, 5, 1), (1, 10, 1)]:
    label = f"a{alpha}_l{lam}_g{gamma}"
    with mlflow.start_run(run_name=f"XGB_reg_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=800,
            max_depth=best_depth,
            min_child_weight=best_mcw,
            learning_rate=0.1,
            subsample=best_subsample,
            colsample_bytree=best_colsample,
            reg_alpha=alpha,
            reg_lambda=lam,
            gamma=gamma,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        best_iter  = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred  = clf.predict_proba(X_tr,  iteration_range=(0, best_iter + 1))[:, 1]
        train_auc  = roc_auc_score(y_tr,  y_tr_pred)
        val_auc    = roc_auc_score(y_val, y_val_pred)
        gap        = train_auc - val_auc
        mlflow.log_params({"reg_alpha": alpha, "reg_lambda": lam, "gamma": gamma,
                           "n_estimators_max": 800})
        mlflow.log_metrics({"train_auc": train_auc, "val_auc": val_auc,
                            "gap": gap, "best_iteration": best_iter})
        reg_results.append({"label": label, "alpha": alpha, "lambda": lam, "gamma": gamma,
                            "val_auc": val_auc, "gap": gap, "best_iter": best_iter})
        converged = "✅ CONVERGED" if best_iter < 799 else "⏩ STILL LEARNING"
        print(f"  {label:<15} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f} {converged}")

reg_df = pd.DataFrame(reg_results)

converged_reg = reg_df[reg_df["best_iter"] < 799]
if len(converged_reg) == 0:
    converged_reg = reg_df

converged_reg = converged_reg.copy()
converged_reg["stability_score"] = (
    converged_reg["val_auc"] - 2 * (converged_reg["gap"] - 0.020).clip(lower=0)
)

best_reg_row = converged_reg.loc[converged_reg["stability_score"].idxmax()]
best_alpha   = best_reg_row["alpha"]
best_lambda  = best_reg_row["lambda"]
best_gamma   = best_reg_row["gamma"]

print(f"\n{'label':<17} | {'val_auc':>8} | {'gap':>6} | {'stability':>10} | converged")
print("-" * 60)
for _, row in converged_reg.sort_values("stability_score", ascending=False).iterrows():
    print(f"  {row['label']:<15} | {row['val_auc']:>8.4f} | {row['gap']:>6.4f} | {row['stability_score']:>10.4f} | ✅")

print(f"\n✅ Best reg: alpha={best_alpha}, lambda={best_lambda}, gamma={best_gamma}")
print(f"   Val AUC={best_reg_row['val_auc']:.4f} | Gap={best_reg_row['gap']:.4f} | Stability={best_reg_row['stability_score']:.4f}")

  a0_l1_g0        | Best Iter: 794 | Val: 0.9784 | Gap: 0.0212 ✅ CONVERGED
🏃 View run XGB_reg_a0_l1_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6b762dad4d884584933d299d9cbb23f6
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a1_l1_g0        | Best Iter: 799 | Val: 0.9788 | Gap: 0.0209 ⏩ STILL LEARNING
🏃 View run XGB_reg_a1_l1_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/e7744ead453445dea2f4c3a5198d9858
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a0_l5_g0        | Best Iter: 797 | Val: 0.9782 | Gap: 0.0214 ✅ CONVERGED
🏃 View run XGB_reg_a0_l5_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/2819aebaf0f34b058584c06f8476a308
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a1_l5_g0        | Best I

In [12]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import mlflow

lr_results = []
lr_n_est_map = {0.01: 5000, 0.05: 2000, 0.1: 1000, 0.2: 500}

for lr, n_est in lr_n_est_map.items():
    with mlflow.start_run(run_name=f"XGB_lr_{lr}"):
        clf = xgb.XGBClassifier(
            n_estimators=n_est,
            max_depth=best_depth,
            min_child_weight=best_mcw,
            learning_rate=lr,
            subsample=best_subsample,
            colsample_bytree=best_colsample,
            reg_alpha=best_alpha,
            reg_lambda=best_lambda,
            gamma=best_gamma,
            scale_pos_weight=spw,
            tree_method="hist",
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=100,
            random_state=42
        )
        
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter = getattr(clf, "best_iteration", n_est - 1)
        
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred  = clf.predict_proba(X_tr,  iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr,  y_tr_pred)
        val_auc   = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_params({"learning_rate": lr, "n_estimators_final": best_iter + 1})
        mlflow.log_metrics({
            "train_auc": train_auc, 
            "val_auc": val_auc,
            "gap": gap, 
            "best_iteration": best_iter
        })

        status = "✅" if best_iter < (n_est - 1) else "⚠️ HIT MAX N"
        
        lr_results.append({
            "lr": lr, "n_est_max": n_est, "best_iter": best_iter,
            "train_auc": train_auc, "val_auc": val_auc, "gap": gap
        })
        
        print(f"lr={lr:<5} | Best Iter: {best_iter:<4} | Val: {val_auc:.4f} | Gap: {gap:.4f} {status}")

lr_df = pd.DataFrame(lr_results)

valid_lr = lr_df[lr_df["gap"] <= 0.035].copy()
if valid_lr.empty:
    print("Warning: No runs met the gap threshold. Using best available.")
    valid_lr = lr_df.copy()

best_lr_row = valid_lr.loc[valid_lr["val_auc"].idxmax()]
best_lr     = float(best_lr_row["lr"])
best_n_est  = int(best_lr_row["best_iter"]) + 1

print(f"\n🏆 Final Champion Parameters:")
print(f"   Learning Rate: {best_lr}")
print(f"   N Estimators:  {best_n_est}")
print(f"   Validation AUC: {best_lr_row['val_auc']:.5f}")

lr=0.01  | Best Iter: 4999 | Val: 0.9774 | Gap: 0.0211 ⚠️ HIT MAX N
🏃 View run XGB_lr_0.01 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/bcfadf39d28a4242a7f1fd2793877f86
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
lr=0.05  | Best Iter: 1690 | Val: 0.9791 | Gap: 0.0206 ✅
🏃 View run XGB_lr_0.05 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/a4bf7ee3f31544748cac254fba65c135
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
lr=0.1   | Best Iter: 833  | Val: 0.9789 | Gap: 0.0208 ✅
🏃 View run XGB_lr_0.1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/e97515120b4c4cb39348145d588c17f0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
lr=0.2   | Best Iter: 497  | Val: 0.9779 | Gap: 0.0220 ✅
🏃 View run XGB_lr_0.2 at: https:/

In [13]:
print(f"best_depth: {best_depth}")
print(f"best_mcw: {best_mcw}")
print(f"best_lr: {best_lr}")
print(f"best_subsample: {best_subsample}")
print(f"best_colsample: {best_colsample}")
print(f"best_alpha: {best_alpha}")
print(f"best_lambda: {best_lambda}")
print(f"best_gamma: {best_gamma}")
print(f"spw: {spw}")
print(f"best_n_est: {best_n_est}")

best_depth: 10
best_mcw: 50
best_lr: 0.05
best_subsample: 1.0
best_colsample: 1.0
best_alpha: 1
best_lambda: 5
best_gamma: 1
spw: 27.58
best_n_est: 1691


In [14]:
from mlflow.models.signature import infer_signature
import numpy as np

with mlflow.start_run(run_name="XGB_CrossValidation_Full"):
    final_params = {
        "max_depth":         int(best_depth),
        "min_child_weight":  int(best_mcw),
        "learning_rate":     float(best_lr),
        "subsample":         float(best_subsample),
        "colsample_bytree":  float(best_colsample),
        "reg_alpha":         float(best_alpha),
        "reg_lambda":        float(best_lambda),
        "gamma":             float(best_gamma),
        "scale_pos_weight":  float(spw),
        "tree_method":       "hist",
        "device":            "cuda",
        "random_state":      42
    }

    MAX_EST = 5000 
    
    mlflow.log_params(final_params)
    mlflow.log_param("cv_folds", 5)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    cv_train_aucs = []
    cv_iterations = []

    print(f"🚀 Running 5-fold CV on {len(X_train_final_xgb):,} rows")

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_final_xgb, y_train_fe_xgb)):
        X_fold_tr, X_fold_val = X_train_final_xgb.iloc[train_idx], X_train_final_xgb.iloc[val_idx]
        y_fold_tr, y_fold_val = y_train_fe_xgb.iloc[train_idx], y_train_fe_xgb.iloc[val_idx]

        model = xgb.XGBClassifier(
            n_estimators=MAX_EST, 
            **final_params, 
            eval_metric="auc",
            early_stopping_rounds=100
        )
        
        model.fit(X_fold_tr, y_fold_tr, eval_set=[(X_fold_val, y_fold_val)], verbose=False)

        best_f_iter = model.best_iteration
        cv_iterations.append(best_f_iter)

        val_probs = model.predict_proba(X_fold_val, iteration_range=(0, best_f_iter + 1))[:, 1]
        tr_probs  = model.predict_proba(X_fold_tr,  iteration_range=(0, best_f_iter + 1))[:, 1]
        
        val_score = roc_auc_score(y_fold_val, val_probs)
        train_score = roc_auc_score(y_fold_tr, tr_probs)

        cv_scores.append(val_score)
        cv_train_aucs.append(train_score)

        mlflow.log_metric("fold_val_auc", round(val_score, 5), step=fold)
        mlflow.log_metric("fold_best_iter", best_f_iter, step=fold)
        
        print(f"  Fold {fold+1}: Best Iter: {best_f_iter:<4} | Train: {train_score:.4f} | Val: {val_score:.4f}")

    avg_val = np.mean(cv_scores)
    std_val = np.std(cv_scores)
    avg_train = np.mean(cv_train_aucs)
    avg_iter = np.mean(cv_iterations)
    avg_gap = avg_train - avg_val

    mlflow.log_metrics({
        "cv_mean_val": round(avg_val, 5),
        "cv_std_val": round(std_val, 5),
        "cv_mean_train": round(avg_train, 5),
        "cv_avg_iteration": int(avg_iter)
    })

    print(f"\n✅ CV Mean Val AUC: {avg_val:.4f} (+/- {std_val:.4f})")
    print(f"   CV Avg Iteration: {int(avg_iter)}")
    print(f"   CV Mean Gap:      {avg_gap:.4f}")

🚀 Running 5-fold CV on 590,540 rows
  Fold 1: Best Iter: 1723 | Train: 0.9997 | Val: 0.9779
  Fold 2: Best Iter: 1749 | Train: 0.9997 | Val: 0.9787
  Fold 3: Best Iter: 1713 | Train: 0.9997 | Val: 0.9787
  Fold 4: Best Iter: 1694 | Train: 0.9997 | Val: 0.9802
  Fold 5: Best Iter: 1798 | Train: 0.9997 | Val: 0.9791

✅ CV Mean Val AUC: 0.9789 (+/- 0.0007)
   CV Avg Iteration: 1735
   CV Mean Gap:      0.0208
🏃 View run XGB_CrossValidation_Full at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/eab0ad8c26024bad8a6a8f26b780f8bd
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [15]:
with mlflow.start_run(run_name="XGB_Final_Pipeline") as run:
    X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
        X_train_final_xgb, y_train_fe_xgb,
        test_size=0.2, random_state=42, stratify=y_train_fe_xgb
    )

    neg_f = int((y_tr_f == 0).sum())
    pos_f = int((y_tr_f == 1).sum())
    spw_f = round(neg_f / pos_f, 2)

    final_params = {
        "n_estimators":      best_n_est + 200,  
        "max_depth":         int(best_depth),
        "min_child_weight":  int(best_mcw),
        "learning_rate":     float(best_lr),
        "subsample":         float(best_subsample),
        "colsample_bytree":  float(best_colsample),
        "reg_alpha":         float(best_alpha),
        "reg_lambda":        float(best_lambda),
        "gamma":             float(best_gamma),
        "scale_pos_weight":  spw_f,
        "tree_method":       "hist",
        "device":            "cuda",
        "random_state":      42
    }

    mlflow.log_params(final_params)
    mlflow.log_param("note", "Final model — optimized params with min_child_weight and gamma")

    final_clf = xgb.XGBClassifier(
        **final_params,
        eval_metric="auc",
        early_stopping_rounds=100
    )
    final_clf.fit(X_tr_f, y_tr_f, eval_set=[(X_val_f, y_val_f)], verbose=100)

    best_iter  = final_clf.best_iteration
    y_val_pred = final_clf.predict_proba(X_val_f, iteration_range=(0, best_iter + 1))[:, 1]
    y_tr_pred  = final_clf.predict_proba(X_tr_f,  iteration_range=(0, best_iter + 1))[:, 1]

    val_auc   = roc_auc_score(y_val_f, y_val_pred)
    train_auc = roc_auc_score(y_tr_f,  y_tr_pred)
    gap = train_auc - val_auc

    mlflow.log_metrics({
        "train_auc":           round(train_auc, 5),
        "val_auc":             round(val_auc,   5),
        "overfit_gap":         round(gap, 5),
        "actual_best_iteration": best_iter
    })

    if train_auc > 0.999:
        print("⚠️  WARNING: train_auc ≈ 1.0 — overfitting! გაზარდე min_child_weight ან gamma.")
    elif gap > 0.035:
        print(f"⚠️  Gap={gap:.4f} > 0.035 — ზომიერი overfitting")
    else:
        print(f"✅ Gap={gap:.4f} — acceptable")

    final_pipe = Pipeline([("clf", final_clf)])

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="xgboost_pipeline",
        registered_model_name="XGBoost_FraudDetection_v2"
    )

    pkl_path = "xgboost_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"\n🚀 Final Model Training Complete!")
    print(f"   Params: LR={final_params['learning_rate']}, Depth={final_params['max_depth']}, "
          f"MCW={final_params['min_child_weight']}, N(best)={best_iter}")
    print(f"   Results → Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"   Run ID: {run.info.run_id}")

[0]	validation_0-auc:0.92053
[100]	validation_0-auc:0.95815
[200]	validation_0-auc:0.96560
[300]	validation_0-auc:0.96992
[400]	validation_0-auc:0.97248
[500]	validation_0-auc:0.97390
[600]	validation_0-auc:0.97526
[700]	validation_0-auc:0.97628
[800]	validation_0-auc:0.97701
[900]	validation_0-auc:0.97736
[1000]	validation_0-auc:0.97778
[1100]	validation_0-auc:0.97807
[1200]	validation_0-auc:0.97836
[1300]	validation_0-auc:0.97859
[1400]	validation_0-auc:0.97872
[1500]	validation_0-auc:0.97892
[1600]	validation_0-auc:0.97895
[1700]	validation_0-auc:0.97906
[1790]	validation_0-auc:0.97906


2026/05/05 14:38:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


⚠️  WARNING: train_auc ≈ 1.0 — overfitting! გაზარდე min_child_weight ან gamma.


2026/05/05 14:38:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost_FraudDetection_v2' already exists. Creating a new version of this model...
2026/05/05 14:38:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection_v2, version 2
Created version '2' of model 'XGBoost_FraudDetection_v2'.


🏃 View run XGB_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/97967f48e2fc4fd29de0f1865f833554
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


NameError: name 'pickle' is not defined